## **Machine Learning Aplicado a las Finanzas** 🚀
### **HW Sesión 2**

Andrés C. Medina Sanhueza

Senior Datascientist Engineer 

anmedinas@gmail.com

Consideraciones Previas

* Tarea es Individual

* Fecha Entrega : Jueves 7 May 23:59 (cualquier commit posterior a esta fecha, se descontaran puntos)


---
## Parte 1 — Estructura del Proyecto *(15 pts)*

### Instrucciones

Crea un repositorio en GitHub llamado `credit_scoring_<tu_apellido>` con la **estructura estándar del curso** (Clase 1).

La estructura esperada es:

```
credit_scoring/
├── data/
│   ├── raw/              ← bankloan.csv va aquí (NO se modifica nunca)
│   └── processed/        ← features WoE transformadas (generadas por el pipeline)
├── notebooks/
│   └── 01_credit_scoring.ipynb   ← orquestador: solo importa de src/, sin lógica inline
├── src/
│   ├── __init__.py
│   ├── data.py           ← carga, validación y feature engineering
│   ├── features.py       ← WoE, IV, binning
│   ├── models.py         ← entrenamiento, GridSearchCV, serialización
│   └── metrics.py        ← AUC, costo asimétrico, scorecard
├── models/
│   └── baseline_v1/
│       ├── model.pkl
│       └── metadata.json
├── reports/
│   └── figures/
├── environment.yml
├── .gitignore
└── README.md
```

### Requisitos mínimos

- `.gitignore` debe excluir: `data/raw/*`, `*.pkl`, `*.h5`, `.env`, `__pycache__/`, `.ipynb_checkpoints/`
- `README.md` debe permitir reproducir el proyecto completo con **3 comandos** (git clone, conda env create, jupyter lab)
- `environment.yml` debe tener versiones fijas para todas las dependencias

### Criterios de evaluación Parte 1

| Criterio | Pts |
|---|---|
| Estructura de carpetas correcta y completa | 5 |
| `.gitignore` excluye datos, modelos y credenciales | 4 |
| `environment.yml` con versiones fijas | 3 |
| `README.md` con instrucciones de reproducción en 3 comandos | 3 |

In [ ]:
# Usa este script para crear la estructura del proyecto automáticamente.
# Ejecútalo UNA VEZ desde la raíz de tu repositorio.
# Luego agrega los archivos .gitignore, environment.yml y README.md manualmente.

from pathlib import Path

def crear_estructura(base: str = ".") -> None:
    directorios = [
        "data/raw",
        "data/processed",
        "notebooks",
        "src",
        "models/baseline_v1",
        "reports/figures",
    ]
    raiz = Path(base)
    for d in directorios:
        (raiz / d).mkdir(parents=True, exist_ok=True)
        (raiz / d / ".gitkeep").touch()
    (raiz / "src" / "__init__.py").touch()
    print("Estructura creada:")
    for p in sorted(raiz.rglob("*")):
        nivel = len(p.relative_to(raiz).parts) - 1
        print("    " * nivel + "├── " + p.name)

# TODO: cambia el path al directorio raíz de tu repositorio
# crear_estructura(base="/ruta/a/tu/credit_scoring")

---
## Parte 2 — Modularización en `src/` *(40 pts)*

Extrae **toda la lógica de negocio** del notebook `MLCreditScoring.ipynb` en los 4 módulos de `src/`.

**Regla general:** cada función debe tener *type hints* y una docstring de una línea que explique el **qué**, no el **cómo**.

---

### 2.1 — `src/data.py` *(8 pts)*

Responsabilidades: cargar el CSV, validar el schema, crear features derivadas.

In [ ]:
# Contenido esperado para src/data.py
# Copia este esqueleto al archivo src/data.py de tu proyecto y completa los TODO

DATA_PY = '''
import pandas as pd

REQUIRED_COLUMNS = [
    "Age", "Employ", "Address", "Income",
    "Creddebt", "OthDebt", "MonthlyLoad", "Default"
]

def load_raw(path: str) -> pd.DataFrame:
    """Carga el dataset crudo desde un CSV."""
    # TODO: lee el CSV y retorna el DataFrame
    pass


def validate_schema(df: pd.DataFrame) -> None:
    """Lanza ValueError si falta alguna columna requerida."""
    # TODO: verifica que todas las columnas de REQUIRED_COLUMNS estén en df
    # Si falta alguna, lanza: ValueError(f"Columnas faltantes: {missing}")
    pass


def create_features(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega features derivadas y elimina filas con NaN."""
    # TODO: crea OthDebtRatio = OthDebt / Income
    # TODO: elimina filas con NaN (dropna)
    # TODO: retorna el DataFrame modificado
    pass
'''

print(DATA_PY)

### 2.2 — `src/features.py` *(12 pts)*

Responsabilidades: cálculo de WoE e IV, selección de variables por IV, transformación WoE.

In [ ]:
# Contenido esperado para src/features.py

FEATURES_PY = '''
import numpy as np
import pandas as pd


def compute_woe_iv(
    df: pd.DataFrame,
    feature: str,
    target: str,
    bins: int = 10
) -> tuple[pd.DataFrame, float]:
    """
    Calcula WoE e IV para una variable numérica continua.

    Returns
    -------
    woe_table : DataFrame con columnas [bin, n_events, n_non_events, woe, iv_bin]
    iv        : Information Value total de la variable
    """
    # TODO: implementa el cálculo de WoE e IV
    # Pasos:
    #   1. Discretiza la variable en `bins` intervalos (pd.qcut o pd.cut)
    #   2. Para cada bin calcula: n_events (target=1), n_non_events (target=0)
    #   3. dist_events = n_events / total_events
    #      dist_non_events = n_non_events / total_non_events
    #   4. woe = ln(dist_events / dist_non_events)  — reemplaza 0 por 0.0001
    #   5. iv_bin = (dist_events - dist_non_events) * woe
    #   6. iv = sum(iv_bin)
    pass


def select_features_by_iv(
    df: pd.DataFrame,
    target: str,
    threshold: float = 0.1
) -> list[str]:
    """Retorna la lista de variables con IV >= threshold."""
    # TODO: itera sobre las columnas numéricas (excluye target)
    # Calcula IV con compute_woe_iv y filtra las que superen el umbral
    pass


def build_woe_tables(
    df: pd.DataFrame,
    features: list[str],
    target: str
) -> dict[str, pd.DataFrame]:
    """Genera el diccionario {feature: woe_table} para las variables seleccionadas."""
    # TODO: retorna {nombre_feature: woe_table} para cada feature en features
    pass


def transform_woe(
    df: pd.DataFrame,
    woe_tables: dict[str, pd.DataFrame]
) -> pd.DataFrame:
    """Reemplaza cada feature por su valor WoE según la tabla de entrenamiento."""
    # TODO: para cada feature en woe_tables, mapea el valor original al WoE del bin correspondiente
    # Retorna un DataFrame con columnas renombradas a <feature>_woe
    pass
'''

print(FEATURES_PY)

### 2.3 — `src/models.py` *(12 pts)*

Responsabilidades: entrenamiento con GridSearchCV, evaluación comparativa, serialización con metadata.

In [ ]:
# Contenido esperado para src/models.py

MODELS_PY = '''
import json
import pickle
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

SEED = 42

MODELOS_CONFIG = {
    "Random Forest": (
        RandomForestClassifier(random_state=SEED),
        {"n_estimators": [100, 200], "max_depth": [4, 6, None]},
    ),
    "XGBoost": (
        XGBClassifier(random_state=SEED, eval_metric="logloss", verbosity=0),
        {"n_estimators": [100, 200], "max_depth": [3, 5], "learning_rate": [0.05, 0.1]},
    ),
    "Logistic Regression": (
        LogisticRegression(random_state=SEED, max_iter=1000),
        {"C": [0.01, 0.1, 1, 10]},
    ),
}


def train_all_models(
    X_train: np.ndarray,
    y_train: np.ndarray
) -> dict:
    """Entrena todos los modelos de MODELOS_CONFIG con GridSearchCV y retorna los mejores estimadores."""
    # TODO: itera sobre MODELOS_CONFIG
    # Para cada modelo: ajusta GridSearchCV(cv=5, scoring="roc_auc", n_jobs=-1)
    # Retorna dict {nombre: gs.best_estimator_}
    pass


def evaluate_models(
    models: dict,
    X_test: np.ndarray,
    y_test: np.ndarray
) -> pd.DataFrame:
    """Retorna DataFrame con AUC-ROC de cada modelo ordenado de mayor a menor."""
    # TODO: calcula roc_auc_score para cada modelo en el test set
    # Retorna pd.DataFrame con columnas ["Modelo", "AUC"] ordenado descendente
    pass


def save_model(
    model,
    path: str,
    metadata: dict
) -> None:
    """Serializa el modelo en .pkl y guarda metadata.json en el mismo directorio."""
    # TODO:
    # 1. Crea el directorio si no existe (Path(path).mkdir(parents=True, exist_ok=True))
    # 2. Guarda el modelo como model.pkl con pickle
    # 3. Agrega "saved_at": date.today().isoformat() al dict metadata
    # 4. Escribe metadata.json con json.dumps(..., indent=2)
    pass
'''

print(MODELS_PY)

### 2.4 — `src/metrics.py` *(8 pts)*

Responsabilidades: AUC-ROC, función de costo asimétrico, construcción del scorecard.

In [ ]:
# Contenido esperado para src/metrics.py

METRICS_PY = '''
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score


def auc_roc(model, X: np.ndarray, y: np.ndarray) -> float:
    """Retorna el AUC-ROC del modelo sobre (X, y)."""
    # TODO
    pass


def costo_total(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    umbral: float,
    c_fn: float = 500,
    c_fp: float = 100
) -> float:
    """
    Calcula el costo operacional total dado un umbral de clasificación.

    c_fn : costo de Falso Negativo (aprueba a quien no pagará)
    c_fp : costo de Falso Positivo (rechaza a quien sí pagaría)
    """
    # TODO
    # y_pred = (y_prob >= umbral).astype(int)
    # fn = ...
    # fp = ...
    # return fn * c_fn + fp * c_fp
    pass


def build_scorecard(
    model_lr,
    woe_tables: dict,
    base_score: int = 300,
    pdo: int = 50
) -> pd.DataFrame:
    """
    Convierte los coeficientes de regresión logística en puntos de scorecard.

    Parámetros
    ----------
    model_lr   : LogisticRegression entrenado sobre variables WoE
    woe_tables : dict {feature: woe_table} con columnas [bin, woe]
    base_score : puntaje base (por defecto 300)
    pdo        : puntos para duplicar las chances (por defecto 50)

    Returns
    -------
    DataFrame con columnas [feature, bin, woe, puntos]
    """
    # TODO: implementa la conversión de coeficientes a puntos
    # factor = pdo / np.log(2)
    # offset = base_score - factor * np.log(odds_base)
    # puntos_bin = -(coef * woe + intercepto/n_vars) * factor
    pass
'''

print(METRICS_PY)

### Criterios de evaluación Parte 2

| Módulo | Criterio | Pts |
|---|---|---|
| `data.py` | `validate_schema` lanza `ValueError` con columnas faltantes; `create_features` es pura (no modifica el df original) | 8 |
| `features.py` | `compute_woe_iv` maneja divisiones por cero; `transform_woe` usa las tablas del train (no recalcula en test) | 12 |
| `models.py` | `train_all_models` retorna el mejor estimador por GridSearchCV; `save_model` genera `.pkl` y `metadata.json` | 12 |
| `metrics.py` | `costo_total` es vectorizada (sin loops); `build_scorecard` retorna un DataFrame interpretable | 8 |

---
## Parte 3 — Notebook Orquestador *(20 pts)*

El archivo `notebooks/01_credit_scoring.ipynb` debe ser un **orquestador puro**: importa funciones de `src/` y muestra resultados. No debe contener lógica de negocio inline.

### Regla de oro

> Si alguien borra la carpeta `src/`, el notebook debe fallar con `ImportError` — **no** con `NameError` por código duplicado en celdas.

### Estructura esperada del notebook orquestador

El notebook debe seguir este flujo en celdas separadas:

In [ ]:
# Celda 1 — Imports (así debe verse en tu notebook orquestador)
# from src.data import load_raw, validate_schema, create_features
# from src.features import select_features_by_iv, build_woe_tables, transform_woe
# from src.models import train_all_models, evaluate_models, save_model
# from src.metrics import auc_roc, costo_total, build_scorecard

print("[Esqueleto — implementa en notebooks/01_credit_scoring.ipynb]")

In [ ]:
# Celda 2 — Carga y validación
# df = load_raw("../data/raw/bankloan.csv")
# validate_schema(df)
# df = create_features(df)

print("[Esqueleto — implementa en notebooks/01_credit_scoring.ipynb]")

In [ ]:
# Celda 3 — Split y selección de features por IV
# from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(
#     df.drop(columns=["Default"]), df["Default"],
#     test_size=0.2, random_state=42, stratify=df["Default"]
# )
# features_sel = select_features_by_iv(X_train.assign(Default=y_train), target="Default", threshold=0.1)
# print(f"Features seleccionadas (IV >= 0.1): {features_sel}")

print("[Esqueleto — implementa en notebooks/01_credit_scoring.ipynb]")

In [ ]:
# Celda 4 — Transformación WoE
# woe_tables = build_woe_tables(X_train.assign(Default=y_train), features_sel, target="Default")
# X_train_woe = transform_woe(X_train[features_sel], woe_tables)
# X_test_woe  = transform_woe(X_test[features_sel],  woe_tables)   # usa tablas del train

print("[Esqueleto — implementa en notebooks/01_credit_scoring.ipynb]")

In [ ]:
# Celda 5 — Entrenamiento y evaluación
# models = train_all_models(X_train_woe, y_train)
# tabla_auc = evaluate_models(models, X_test_woe, y_test)
# display(tabla_auc)

print("[Esqueleto — implementa en notebooks/01_credit_scoring.ipynb]")

In [ ]:
# Celda 6 — Serialización del mejor modelo
# mejor_nombre = tabla_auc.iloc[0]["Modelo"]
# mejor_modelo  = models[mejor_nombre]

# metadata = {
#     "model_name":        mejor_nombre,
#     "version":           "1.0",
#     "author":            "tu_apellido",
#     "dataset":           "data/raw/bankloan.csv",
#     "n_train_samples":   int(len(y_train)),
#     "n_test_samples":    int(len(y_test)),
#     "features_selected": features_sel,
#     "hyperparameters":   mejor_modelo.get_params(),
#     "metrics": {
#         "auc_test": round(auc_roc(mejor_modelo, X_test_woe, y_test), 4),
#         "costo_total_test": int(costo_total(y_test.values, mejor_modelo.predict_proba(X_test_woe)[:, 1], umbral=0.5)),
#     },
# }

# save_model(mejor_modelo, path="../models/baseline_v1", metadata=metadata)
# print("Modelo serializado ✓")

print("[Esqueleto — implementa en notebooks/01_credit_scoring.ipynb]")

### Criterios de evaluación Parte 3

| Criterio | Pts |
|---|---|
| El notebook corre de arriba abajo sin errores (`Restart & Run All`) | 8 |
| No contiene lógica de negocio inline (WoE, bucles de entrenamiento, etc.) | 7 |
| Las celdas siguen el flujo: carga → features → split → WoE → train → evaluate → save | 5 |

---
## Parte 4 — Git con Conventional Commits *(15 pts)*

El historial de tu repositorio debe mostrar **al menos 8 commits** que sigan la convención del curso (Clase 1).

### Historial mínimo esperado

```bash
git log --oneline
```

```
a1b2c3d  exp(models): compara 3 modelos — XGBoost AUC 0.82 en test
b2c3d4e  refac(notebook): convierte notebook a orquestador puro
c3d4e5f  feat(src): métricas financieras y scorecard en metrics.py
d4e5f6a  feat(src): entrenamiento y serialización de modelos en models.py
e5f6a7b  feat(src): cálculo WoE/IV y transformación en features.py
f6a7b8c  feat(src): carga, validación y feature engineering en data.py
a7b8c9d  data(raw): agrega bankloan.csv
b8c9d0e  feat: estructura inicial del proyecto
```

### Regla de oro

> Un solo commit con todo el trabajo = **0 puntos** en esta parte.

### Tipos de commit válidos para el curso

| Prefijo | Cuándo usarlo |
|---|---|
| `feat:` | Nueva funcionalidad o módulo |
| `fix:` | Corrección de bug (especialmente: data leakage, errores de validación) |
| `exp:` | Experimento: nuevo modelo, hiperparámetro, resultado |
| `data:` | Cambio en datos o pipeline de datos |
| `refac:` | Refactorización sin cambio de comportamiento |
| `docs:` | README, docstrings, comentarios |

### Criterios de evaluación Parte 4

| Criterio | Pts |
|---|---|
| Al menos 8 commits en el historial | 5 |
| Todos los commits siguen la convención `tipo(scope): descripción` | 5 |
| Los mensajes son descriptivos (incluyen métrica, resultado o motivo) | 5 |

---
## Parte 5 — `metadata.json` Automático *(10 pts)*

Al llamar a `save_model()`, debe generarse automáticamente `models/baseline_v1/metadata.json`.

### Formato esperado

```json
{
  "model_name": "XGBoost",
  "version": "1.0",
  "saved_at": "2026-05-10",
  "author": "tu_apellido",
  "dataset": "data/raw/bankloan.csv",
  "n_train_samples": 560,
  "n_test_samples": 140,
  "features_selected": ["Age_woe", "Employ_woe", "Income_woe"],
  "hyperparameters": {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.1},
  "metrics": {
    "auc_test": 0.82,
    "costo_total_test": 4200
  }
}
```

### Requisitos

- El campo `saved_at` debe generarse automáticamente con `date.today().isoformat()` — no hardcodeado
- `hyperparameters` debe venir de `model.get_params()` — no hardcodeado
- El archivo **no debe estar trackeado en Git** (`.gitignore` excluye `models/baseline_v1/`)

### Verificación en el notebook orquestador

In [ ]:
# Celda de verificación — agrégala al final del notebook orquestador
import json
from pathlib import Path

metadata_path = Path("../models/baseline_v1/metadata.json")

assert metadata_path.exists(), "metadata.json no existe — revisa save_model()"

with open(metadata_path) as f:
    meta = json.load(f)

campos_requeridos = [
    "model_name", "version", "saved_at", "author",
    "dataset", "n_train_samples", "n_test_samples",
    "features_selected", "hyperparameters", "metrics"
]

for campo in campos_requeridos:
    assert campo in meta, f"Campo faltante en metadata.json: {campo}"

print("metadata.json válido ✓")
print(json.dumps(meta, indent=2))

### Criterios de evaluación Parte 5

| Criterio | Pts |
|---|---|
| `metadata.json` se genera automáticamente al llamar `save_model()` | 4 |
| `saved_at` es dinámico y `hyperparameters` viene de `get_params()` | 3 |
| La celda de verificación pasa sin errores | 3 |

---
## Checklist de Entrega

Antes de enviar la URL del repositorio, verifica que:

- [ ] El repositorio es público en GitHub y la URL es accesible
- [ ] La estructura de carpetas coincide exactamente con la especificada en la Parte 1
- [ ] `environment.yml` tiene versiones fijas (no `>=` sin pin de versión mayor)
- [ ] `README.md` tiene los 3 comandos de reproducción
- [ ] `src/data.py`, `src/features.py`, `src/models.py` y `src/metrics.py` existen y no tienen `pass` pendientes
- [ ] `notebooks/01_credit_scoring.ipynb` corre de arriba abajo con `Restart & Run All` sin errores
- [ ] No hay lógica de negocio en el notebook orquestador (sin cálculos WoE, entrenamiento, etc. inline)
- [ ] `git log --oneline` muestra al menos 8 commits con conventional commits
- [ ] La celda de verificación de `metadata.json` pasa sin errores
- [ ] `data/raw/bankloan.csv` **no está en el repositorio** (excluido por `.gitignore`)

---

**Archivo a entregar:** URL del repositorio GitHub enviada por mail a `anmedinas@gmail.com`  
**Asunto del mail:** `[HW03] credit_scoring_<tu_apellido>`